# Manager Report - B2B BookStore, 2025 vs 2024

One-page view of how the business did in 2025 compared with 2024, built from `book_sales_clean.xlsx`.
Sections: **KPIs**, **monthly trend**, **sales and growth by customer type, region and category**, and **findings** (finding, cause, action).

Run the **Setup** cell first. Hover over any chart to see exact values.

## Setup

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

df = pd.read_excel("book_sales_clean.xlsx")
df["GrossSale"] = df["Quantity"] * df["UnitPrice"]
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month

sales = df[~df["IsReturn"]]
returns = df[df["IsReturn"]]
YEARS = [2024, 2025]

# ---- Chart style (same palette as analysis.ipynb) -------------------------
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#b5b4ae"
GOOD, BAD = "#1baf7a", "#e34948"
TEXT, TEXT_2, GRID, BG = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
YEAR_COLORS = {2024: GREY, 2025: BLUE}


def style(fig, title, height=420, legend=False):
    """Apply one consistent, quiet look to every chart."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=TEXT), x=0, xanchor="left"),
        template="plotly_white", height=height,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=TEXT_2),
        plot_bgcolor=BG, paper_bgcolor=BG,
        margin=dict(l=10, r=30, t=70, b=10), bargap=0.3, showlegend=legend,
        legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, title_text=""),
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, title_font_size=12)
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, title_font_size=12)
    fig.update_traces(selector=dict(type="bar"), marker_cornerradius=4)
    return fig


def money(x):
    """Short money label: 12.3M / 456K."""
    return f"{x/1e6:,.1f}M" if abs(x) >= 1e6 else f"{x/1e3:,.0f}K"


def kpis(d):
    """The eight headline KPIs for one slice of the data."""
    s, r = d[~d["IsReturn"]], d[d["IsReturn"]]
    return pd.Series({
        "Gross sales": s["GrossSale"].sum(),
        "Net sales": d["NetSale"].sum(),
        "Avg discount rate (%)": s["DiscountRate"].mean() * 100,
        "Return rate (%)": -r["NetSale"].sum() / s["NetSale"].sum() * 100,
        "Active customers": d["CustomerID"].nunique(),
        "Average order value": s.groupby("InvoiceNo")["NetSale"].sum().mean(),
        "Orders": s["InvoiceNo"].nunique(),
        "Units": s["Quantity"].sum(),
    })

## 1. KPIs

**Brief:** Net sales grew **7.5%** to 67.4M, on **18% more orders** and 10% more units.
But each order got smaller (average order value **-9%**), discounts rose (23.4% -> 24.5%), so gross sales grew faster (+9.3%) than net sales.
Returns stay small (0.60% of sales). Active customers fell slightly over the year (120 -> 117), but **no new customers** joined in 2025.

Definitions: *Gross sales* = quantity x unit price, before discount (sales only). *Net sales* = after discounts and returns.
*Avg discount rate* = average of the line discount rates. *Return rate* = returned value / sales value.
*Orders* = sales invoices. *Average order value* = net sales per sales invoice. *Active customers* = customers with at least one order in the year.

In [2]:
kpi = pd.DataFrame({y: kpis(df[df["Year"] == y]) for y in YEARS})
kpi["Change %"] = (kpi[2025] / kpi[2024] - 1) * 100
kpi["2024-2025 total"] = kpis(df)

# Rates are shown with 2 decimals, everything else as whole numbers
rate_rows = ["Avg discount rate (%)", "Return rate (%)"]
fmt = kpi.astype(object)
for c in [2024, 2025, "2024-2025 total"]:
    fmt[c] = [f"{v:.2f}%" if i in rate_rows else f"{v:,.0f}" for i, v in kpi[c].items()]
fmt["Change %"] = kpi["Change %"].map(lambda v: f"{v:+.1f}%")
display(fmt.rename(index=lambda i: i.replace(" (%)", "")))

,2024,2025,Change %,2024-2025 total
Gross sales,"79,597,625","86,980,165",+9.3%,"166,577,790"
Net sales,"62,734,326","67,430,914",+7.5%,"130,165,240"
Avg discount rate,23.42%,24.52%,+4.7%,24.01%
Return rate,0.54%,0.60%,+11.5%,0.57%
Active customers,120,117,-2.5%,120
Average order value,"23,301","21,200",-9.0%,"22,163"
Orders,"2,707","3,200",+18.2%,"5,907"
Units,"371,050","408,070",+10.0%,"779,120"


In [3]:
# KPI cards: 2025 value, with the change vs 2024 underneath.
# Green = good direction, red = bad direction (for discount and return rate, going up is bad).
cards = [  # (KPI, label, prefix/suffix scale, higher is better)
    ("Gross sales", "Gross sales", "M", True),
    ("Net sales", "Net sales", "M", True),
    ("Avg discount rate (%)", "Avg discount rate", "%", False),
    ("Return rate (%)", "Return rate", "%", False),
    ("Active customers", "Active customers", "", True),
    ("Average order value", "Average order value", "K", True),
    ("Orders", "Orders", "", True),
    ("Units", "Units", "", True),
]
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2, vertical_spacing=0.2)
for i, (key, label, unit, up_good) in enumerate(cards):
    scale = {"M": 1e6, "K": 1e3}.get(unit, 1)
    good, bad = (GOOD, BAD) if up_good else (BAD, GOOD)
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=kpi.loc[key, 2025] / scale,
        number=dict(suffix=unit, valueformat=".2f" if unit == "%" else (".1f" if unit else ",.0f"),
                    font=dict(size=30, color=TEXT)),
        delta=dict(reference=kpi.loc[key, 2024] / scale, relative=True, valueformat="+.1%",
                   increasing=dict(color=good), decreasing=dict(color=bad), font=dict(size=14)),
        title=dict(text=f"{label}<br><span style='font-size:11px;color:{TEXT_2}'>2025 vs 2024</span>",
                   font=dict(size=13, color=TEXT_2)),
    ), row=i // 4 + 1, col=i % 4 + 1)
style(fig, "2025 KPIs: more orders and sales, but smaller orders and higher discounts", height=330)
fig.show()

## 2. Monthly trend

**Brief:** 2025 ran well ahead of 2024 from January to July (+9% to +61% each month).
From August it lost that lead: **August -21%**, **October -23%**, and November-December flat.
August-September is the back-to-school peak, so a weaker August hits the year hard.

In [4]:
monthly = df.pivot_table(index="Month", columns="Year", values="NetSale", aggfunc="sum")
monthly["Growth %"] = (monthly[2025] / monthly[2024] - 1) * 100
months = pd.to_datetime(monthly.index.astype(str), format="%m").strftime("%b")

display(monthly.set_axis(months).style.format({2024: "{:,.0f}", 2025: "{:,.0f}", "Growth %": "{:+.1f}%"}))

fig = go.Figure()
for y in YEARS:
    fig.add_scatter(x=months, y=monthly[y], name=str(y), mode="lines+markers",
                    line=dict(color=YEAR_COLORS[y], width=2),
                    marker=dict(size=8, color=YEAR_COLORS[y], line=dict(color=BG, width=2)),
                    hovertemplate=f"{y}: " + "%{y:,.0f}<extra></extra>")
# Direct labels where the two lines are far apart, plus the two weakest months
for y, shift in [(2025, 14), (2024, -14)]:
    fig.add_annotation(x="Apr", y=monthly.loc[4, y], text=f"<b>{y}</b>", showarrow=False, yshift=shift,
                       font=dict(color=TEXT))
for m in [8, 10]:
    fig.add_annotation(x=months[m - 1], y=monthly.loc[m, 2025], text=f"{monthly.loc[m, 'Growth %']:+.0f}% vs 2024",
                       showarrow=True, arrowhead=0, ax=30, ay=40, xanchor="left", font=dict(color=TEXT_2, size=11),
                       arrowcolor=GREY)
style(fig, "Monthly net sales: 2025 led until July, then fell to or below 2024", height=420, legend=True)
fig.update_layout(hovermode="x unified")
fig.update_yaxes(title="Net sales", rangemode="tozero")
fig.update_xaxes(title=None, showgrid=False)
fig.show()

Year,2024,2025,Growth %
Month,,,
Jan,"3,067,676","4,448,797",+45.0%
Feb,"3,029,724","4,110,985",+35.7%
Mar,"3,116,222","4,598,404",+47.6%
Apr,"3,903,335","6,294,634",+61.3%
May,"3,681,871","4,015,894",+9.1%
Jun,"2,501,770","3,088,178",+23.4%
Jul,"2,528,964","3,466,578",+37.1%
Aug,"10,727,670","8,451,587",-21.2%
Sep,"10,163,667","10,429,666",+2.6%


## 3. Sales and growth by customer type, region and category

Each group is ranked by total net sales (2024 + 2025). The left panel compares the two years, the right panel shows 2025 growth
(blue = grew, orange = shrank).

In [5]:
def breakdown(dim, title):
    """Table + chart of net sales by year and growth for one dimension, ranked by total sales."""
    t = df.pivot_table(index=dim, columns="Year", values="NetSale", aggfunc="sum")
    t["Total"] = t[YEARS].sum(axis=1)
    t["Share %"] = t["Total"] / t["Total"].sum() * 100
    t["Growth %"] = (t[2025] / t[2024] - 1) * 100
    t = t.sort_values("Total", ascending=False)
    display(t.style.format({2024: "{:,.0f}", 2025: "{:,.0f}", "Total": "{:,.0f}",
                            "Share %": "{:.1f}%", "Growth %": "{:+.1f}%"}))

    order = t.index[::-1]  # largest on top in horizontal bars
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.62, 0.38], horizontal_spacing=0.04,
                        subplot_titles=("Net sales", "Growth 2025 vs 2024"))
    for y in YEARS[::-1]:  # 2025 added first so 2024 sits on top in each pair
        fig.add_bar(y=order, x=t.loc[order, y], name=str(y), orientation="h", marker_color=YEAR_COLORS[y],
                    text=[money(v) for v in t.loc[order, y]], textposition="outside", cliponaxis=False,
                    hovertemplate=f"{y} " + "%{y}: %{x:,.0f}<extra></extra>", row=1, col=1)
    g = t.loc[order, "Growth %"]
    fig.add_bar(y=order, x=g, orientation="h", showlegend=False,
                marker_color=[BLUE if v >= 0 else ORANGE for v in g],
                text=[f"{v:+.1f}%" for v in g], textposition="outside", cliponaxis=False,
                hovertemplate="%{y}: %{x:+.1f}%<extra></extra>", row=1, col=2)
    style(fig, title, height=140 + 70 * len(t), legend=True)
    fig.update_layout(barmode="group", bargap=0.25, bargroupgap=0.08, margin=dict(t=100),
                      legend=dict(traceorder="reversed"))
    fig.update_annotations(font=dict(size=12, color=TEXT_2))
    fig.update_xaxes(row=1, col=1, title=None, range=[0, t[YEARS].max().max() * 1.2])
    pad = max(5, g.abs().max() * 0.45)
    fig.update_xaxes(row=1, col=2, title=None, ticksuffix="%", zeroline=True, zerolinecolor=TEXT_2,
                     range=[min(0, g.min()) - pad, max(0, g.max()) + pad])
    fig.update_yaxes(showgrid=False)
    fig.show()
    return t

### 3.1 Customer type

**Brief:** Schools are **62% of sales** but did not grow (-1.3%). All the growth came from **Bookstores (+31.7%)**
and **Online Retailers (+22.1%)**, the groups with the highest discounts (~30-34% vs ~15% for schools).

In [6]:
by_type = breakdown("CustomerType", "Customer type: schools are flat, bookstores and online retailers drive growth")

Year,2024,2025,Total,Share %,Growth %
CustomerType,,,,,
School,"40,731,492","40,211,045","80,942,537",62.2%,-1.3%
Bookstore,"10,631,899","14,006,952","24,638,851",18.9%,+31.7%
Online Retailer,"8,644,114","10,552,557","19,196,671",14.7%,+22.1%
Library,"2,726,822","2,660,360","5,387,182",4.1%,-2.4%


### 3.2 Region

**Brief:** **Marmara** is the largest region (41% of sales, +9.7%). **Aegean** grew fastest (+22.3%).
**Central Anatolia (-1.9%)** and **Mediterranean (-8.1%)** declined.

In [7]:
by_region = breakdown("Region", "Region: Aegean grows fastest, Mediterranean and Central Anatolia decline")

Year,2024,2025,Total,Share %,Growth %
Region,,,,,
Marmara,"25,118,406","27,552,600","52,671,006",40.5%,+9.7%
Central Anatolia,"18,685,429","18,332,816","37,018,245",28.4%,-1.9%
Aegean,"13,619,921","16,662,479","30,282,400",23.3%,+22.3%
Mediterranean,"5,310,571","4,883,018","10,193,589",7.8%,-8.1%


### 3.3 Category

**Brief:** **Education** is the largest category (32% of sales) but grew only 2.5%, and **Science** was flat (-0.1%);
both depend mostly on schools. **Business (+21.7%)** and **Fiction (+17.3%)** grew fastest.

In [8]:
by_category = breakdown("Category", "Category: Business and Fiction grow fastest, Education and Science stall")

Year,2024,2025,Total,Share %,Growth %
Category,,,,,
Education,"20,833,154","21,356,099","42,189,253",32.4%,+2.5%
Language Learning,"9,752,284","10,952,508","20,704,791",15.9%,+12.3%
Science,"10,179,322","10,172,993","20,352,314",15.6%,-0.1%
Fiction,"6,967,707","8,174,580","15,142,286",11.6%,+17.3%
Children,"5,996,498","6,396,175","12,392,673",9.5%,+6.7%
History,"4,876,186","5,353,636","10,229,823",7.9%,+9.8%
Business,"4,129,176","5,024,923","9,154,099",7.0%,+21.7%


## 4. Findings

The table below the findings shows the evidence: active customers per quarter by customer type, and the customers
who have not ordered since June 2025.

In [9]:
# Evidence 1: active customers per quarter, by customer type
active = df.pivot_table(index="CustomerType", columns=df["InvoiceDate"].dt.to_period("Q").astype(str),
                        values="CustomerID", aggfunc="nunique")
active.loc["All customers"] = df.groupby(df["InvoiceDate"].dt.to_period("Q").astype(str))["CustomerID"].nunique()
print("Active customers per quarter")
display(active)

# Evidence 2: customers with no order since June 2025, and their share of 2024-2025 sales
last = df.groupby(["CustomerID", "CustomerType"]).agg(LastOrder=("InvoiceDate", "max"), NetSales=("NetSale", "sum")).reset_index()
lapsed = last[last["LastOrder"] < "2025-07-01"]
new_2025 = set(df.loc[df["Year"] == 2025, "CustomerID"]) - set(df.loc[df["Year"] == 2024, "CustomerID"])
print(f"Customers with no order since June 2025: {len(lapsed)} "
      f"({lapsed['NetSales'].sum() / last['NetSales'].sum() * 100:.1f}% of 2024-2025 net sales)")
print(f"New customers in 2025: {len(new_2025)}")
display(lapsed["CustomerType"].value_counts().rename("Lapsed customers").to_frame())

# Evidence 3: school sales in the back-to-school months (August + October)
school_peak = df[(df["CustomerType"] == "School") & df["Month"].isin([8, 10])].groupby("Year")["NetSale"].sum()
print(f"School net sales in Aug + Oct: 2024 {money(school_peak[2024])} -> 2025 {money(school_peak[2025])} "
      f"({(school_peak[2025] / school_peak[2024] - 1) * 100:+.0f}%)")

Active customers per quarter


InvoiceDate,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4
CustomerType,,,,,,,,
Bookstore,40,44,49,50,50,47,40,40
Library,15,15,19,19,18,15,14,12
Online Retailer,16,18,19,20,19,18,15,14
School,25,28,29,28,30,29,24,22
All customers,96,105,116,117,117,109,93,88


Customers with no order since June 2025: 27 (14.5% of 2024-2025 net sales)
New customers in 2025: 0


,Lapsed customers
CustomerType,
Bookstore,10
School,6
Library,6
Online Retailer,5


School net sales in Aug + Oct: 2024 12.4M -> 2025 8.5M (-31%)


| # | Finding | Cause | Action |
|---|---|---|---|
| 1 | **Growth stopped in the second half of 2025.** Jan-Jul was +9% to +61% per month, then August -21%, October -23%, Nov-Dec flat. | Customers stopped ordering: active customers fell from **117 (Q1) to 88 (Q4)**. **27 customers** have not ordered since June 2025 (~14.5% of 2024-2025 sales), and **no new customers** joined in 2025 to replace them. | Start a **win-back campaign** for the 27 lapsed customers, largest first. Set up a monthly "no order in 90 days" alert so sales reps act before a customer is lost. Add a **new-customer target** for 2026. |
| 2 | **Schools (62% of sales) did not grow (-1.3%)**, and the back-to-school peak was weaker. | Active schools fell from 30 to **22** by Q4 2025. School sales in August + October dropped from **12.4M to 8.5M (-31%)**. | Contact every school **in May-June**, before the 2026 season, with pre-order offers or yearly contracts. Give the top schools a named account manager. This also protects **Education** and **Science**, which depend on schools. |
| 3 | **Orders got smaller and discounts higher.** Orders +18%, but average order value -9% and avg discount rate 23.4% -> 24.5%. Gross sales +9.3% vs net sales +7.5%. | Growth came from **Bookstores and Online Retailers**, which place smaller orders and get the highest discounts (~30-34%). | Review the **discount policy**: link discounts to order size (volume tiers) and set a minimum order value for the highest discount levels. Track net sales, not gross, as the growth target. |
| 4 | **Mediterranean (-8.1%) and Central Anatolia (-1.9%) declined**, while Aegean grew +22.3%. | Active customers in Mediterranean fell from 14 to **10** and in Central Anatolia from 36 to **27** during 2025. | Focus the win-back campaign on these two regions first. Look at what works in Aegean (same customer count, +22% sales) and apply it there. |
| 5 | **Returns are small but rising slightly** (0.54% -> 0.60% of sales). | Not a business risk yet. | No action needed now. Keep it on the KPI list and review if it passes 1%. |